# 기존 feature table ->  로지스틱용 정제

In [20]:
# ============================================================
# 20. Logistic Preprocessing (설명용 로지스틱 준비)
# ============================================================
#
# 목적:
# 1. XGBoost 최종 피처 테이블에서 로지스틱용 피처 분리
# 2. 해석을 방해하는 피처 제거 (결측 패턴, 희귀 변수)
# 3. 연속형 / 이진 변수 구분
#
# 입력:
# - features_final.csv
#
# 출력:
# - feature_table_logistic_base.csv
#
# 주의:
# - sliding window 구조 유지
# - patient-level 정보 보존
# ============================================================


In [21]:
import pandas as pd
import numpy as np

In [22]:
INPUT_PATH = "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/features_final.csv"
OUTPUT_PATH = "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/feature_table_logistic_base.csv"
LABEL_COL = "death_next_24h"

EXCLUDE_COLS = [
    "stay_id",
    "subject_id",
    "hadm_id",
    "observation_hour",
    "observation_start",
    "observation_end"
]

df = pd.read_csv(INPUT_PATH)

In [23]:
df.head()

,stay_id,subject_id,hadm_id,observation_hour,observation_start,observation_end,anchor_age,hr,rr,spo2,...,pressor_next_6h,composite_next_6h,death_next_12h,vent_next_12h,pressor_next_12h,composite_next_12h,death_next_24h,vent_next_24h,pressor_next_24h,composite_next_24h
0,30000831,15726459,22744101,6,2140-04-17 21:26:33,2140-04-18 03:26:33,78,94.333333,26.333333,96.000000,...,0,0,0,0,0,0,0,0,0,0
1,30000831,15726459,22744101,7,2140-04-17 22:26:33,2140-04-18 04:26:33,78,87.500000,26.166667,96.166667,...,0,0,0,0,0,0,0,0,0,0
2,30000831,15726459,22744101,8,2140-04-17 23:26:33,2140-04-18 05:26:33,78,82.500000,25.833333,96.000000,...,0,0,0,0,0,0,0,0,0,0
3,30000831,15726459,22744101,9,2140-04-18 00:26:33,2140-04-18 06:26:33,78,82.166667,25.000000,95.833333,...,0,0,0,0,0,0,0,0,0,0
4,30000831,15726459,22744101,10,2140-04-18 01:26:33,2140-04-18 07:26:33,78,82.166667,25.833333,95.333333,...,0,0,0,0,0,0,0,0,0,0


In [24]:
df.columns

Index(['stay_id', 'subject_id', 'hadm_id', 'observation_hour',
       'observation_start', 'observation_end', 'anchor_age', 'hr', 'rr',
       'spo2', 'temp', 'sbp', 'dbp', 'mbp', 'lactate', 'creatinine', 'wbc',
       'platelets', 'potassium', 'sodium', 'gcs_eye', 'gcs_verbal',
       'gcs_motor', 'gcs_total', 'urine_ml_6h', 'urine_ml_kg_hr_avg',
       'oliguria_flag', 'lactate_missing', 'gcs_missing_flag',
       'urine_missing_flag', 'hr_mean_6h', 'hr_std_6h', 'hr_min_6h',
       'hr_max_6h', 'sbp_mean_6h', 'sbp_std_6h', 'sbp_min_6h', 'sbp_max_6h',
       'mbp_mean_6h', 'mbp_std_6h', 'mbp_min_6h', 'mbp_max_6h', 'spo2_mean_6h',
       'spo2_std_6h', 'spo2_min_6h', 'spo2_max_6h', 'rr_mean_6h', 'rr_std_6h',
       'rr_min_6h', 'rr_max_6h', 'hr_delta_1h', 'hr_delta_3h', 'hr_slope_3h',
       'sbp_delta_1h', 'sbp_delta_3h', 'sbp_slope_3h', 'mbp_delta_1h',
       'mbp_delta_3h', 'mbp_slope_3h', 'spo2_delta_1h', 'spo2_delta_3h',
       'spo2_slope_3h', 'lactate_delta_1h', 'lactate_delta_3

In [25]:
X = df.drop(columns=EXCLUDE_COLS + [LABEL_COL])
y = df[LABEL_COL]
groups = df["stay_id"]

binary_cols = [c for c in X.columns if X[c].dropna().isin([0, 1]).all()]
continuous_cols = [c for c in X.columns if c not in binary_cols]

In [26]:
rare_cols = [c for c in binary_cols if X[c].mean() < 0.005]
X = X.drop(columns=rare_cols)
binary_cols = [c for c in binary_cols if c not in rare_cols]

out = X.copy()
out[LABEL_COL] = y
out["stay_id"] = groups

out.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)

Saved: /home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/feature_table_logistic_base.csv


In [27]:
# feature_table_logistic_base.csv

In [28]:
logistic_prep = pd.read_csv("/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/feature_table_logistic_base.csv")

In [29]:
logistic_prep.head()

,anchor_age,hr,rr,spo2,temp,sbp,dbp,mbp,lactate,creatinine,...,vent_next_6h,composite_next_6h,vent_next_12h,pressor_next_12h,composite_next_12h,vent_next_24h,pressor_next_24h,composite_next_24h,death_next_24h,stay_id
0,78,94.333333,26.333333,96.000000,37.0,96.333333,60.333333,72.500000,1.4,2.1,...,0,0,0,0,0,0,0,0,0,30000831
1,78,87.500000,26.166667,96.166667,37.0,96.333333,57.833333,70.500000,1.4,2.1,...,0,0,0,0,0,0,0,0,0,30000831
2,78,82.500000,25.833333,96.000000,37.0,95.166667,57.500000,70.000000,1.4,2.3,...,0,0,0,0,0,0,0,0,0,30000831
3,78,82.166667,25.000000,95.833333,37.0,93.333333,58.333333,69.666667,1.4,2.3,...,0,0,0,0,0,0,0,0,0,30000831
4,78,82.166667,25.833333,95.333333,37.0,96.500000,59.333333,71.333333,1.4,2.3,...,0,0,0,0,0,0,0,0,0,30000831
